# Implementation of Finite Elements

This lecture shows how to implement our own finite elements in C++,
and how to use them within the NGSolve language. We implement first order and second order triangular finite elements.

The finite element is implemented in its own package on github: 
[https://github.com/TUWien-ASC/NGS-myfe](https://github.com/TUWien-ASC/NGS-myfe)

* basis functions are implemented in myelement.cpp/hpp
* the transformation from reference-element to physical elements in in mydiffop.hpp
* the connectivity is implemented in the `FESpace` in myfespace.cpp/hpp
* the Python-bindings are in mymodule.cpp

In [8]:
from ngsolve import *
from ngsolve.webgui import Draw
from myfe import *
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2, quad_dominated=False))

We can now create an instance of our own finite element space:

In [9]:
fes = MyFESpace(mesh, secondorder=True, dirichlet="left|bottom|top")

Constructor of MyFESpace
Flags = secondorder = 1
secondorder
dirichlet = 0: 1
1: 3
2: 4


You have chosen second order elements
Update MyFESpace, #vert = 39, #edge = 94


and use it within NGSolve such as the builtin finite element spaces:

In [10]:
print ("ndof = ", fes.ndof)

ndof =  133


In [11]:
gfu = GridFunction(fes)
gfu.Set(x*y)

Draw (gfu)
Draw (grad(gfu)[0], mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

and solve the standard problem:

In [12]:
u,v = fes.TnT()
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(1*v*dx).Assemble()
gfu.vec.data = a.mat.Inverse(fes.FreeDofs())*f.vec
Draw (gfu);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

Draw basis functions:

In [13]:
gfu.vec[:] = 0
gfu.vec[mesh.nv-3] = 1
gfu.vec[fes.ndof-1] = 1
Draw (gfu, order=2);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

Documentation provided in the DocInfo structure is available in Python - help. Look for 'secondorder'.

In [7]:
help (MyFESpace)

Help on class MyFESpace in module myfe:

class MyFESpace(ngsolve.comp.FESpace)
 |  My own FESpace.
 |
 |  My own FESpace provides first and second order triangular elements.
 |
 |  Keyword arguments can be:
 |
 |  order: int = 1
 |    order of finite element space
 |  complex: bool = False
 |    Set if FESpace should be complex
 |  dirichlet: regexpr
 |    Regular expression string defining the dirichlet boundary.
 |    More than one boundary can be combined by the | operator,
 |    i.e.: dirichlet = 'top|right'
 |  dirichlet_bbnd: regexpr
 |    Regular expression string defining the dirichlet bboundary,
 |    i.e. points in 2D and edges in 3D.
 |    More than one boundary can be combined by the | operator,
 |    i.e.: dirichlet_bbnd = 'top|right'
 |  dirichlet_bbbnd: regexpr
 |    Regular expression string defining the dirichlet bbboundary,
 |    i.e. points in 3D.
 |    More than one boundary can be combined by the | operator,
 |    i.e.: dirichlet_bbbnd = 'top|right'
 |  definedon: 

**Exercises:**

Extend MyFESpace by the following elements:

- 1D finite elements (ET_SEGM), as needed for boundary conditions, $P^1$ and $P^2$.
- quadrilateral elements (ET_QUAD), space $Q^1$, use geom.GenerateMesh(quad_dominated=True)
- tetrahedral elements (ET_TET), $P^1$ and $P^2$, test it for 3D domains

Next, implement $P^3$ triangles and $Q^2$ quadrilaterals.